# Geometry-V1 Batch 2B
PREPARED_NOT_EXECUTED; user-run only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
import hashlib,json,os,pathlib,subprocess,sys,zipfile
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V1'
EXECUTION_EXACT='4e39ec28fc2c1f8cc2848c62360f3ca096184658'; RUN_ID='geometry-v1-b2b-4e39ec28fc2c-operational-01'
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/Batch2B')
repo=pathlib.Path('/content/geometry-v1-source'); run_dir=DRIVE_ROOT/RUN_ID
if repo.exists() or run_dir.exists(): raise FileExistsError('create-only')
subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(repo)],check=True)
subprocess.run(['git','checkout','--detach',EXECUTION_EXACT],cwd=repo,check=True)
def verify_checkout():
 head=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); clean=subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
 if head!=EXECUTION_EXACT or clean: raise RuntimeError('identity')
verify_checkout(); subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout()


In [ ]:
# Run only after fresh user authorization.
from google.colab import files,userdata
uploaded=files.upload()
if len(uploaded) not in (1,2): raise ValueError('one or two RGB images')
input_dir=pathlib.Path('/content/geometry-inputs'); input_dir.mkdir(exist_ok=False); paths=[]
for name,data in uploaded.items():
 path=input_dir/pathlib.Path(name).name
 with path.open('xb') as h: h.write(data)
 paths.append(str(path))
root_key=userdata.get('CEG_WM_ROOT_KEY'); hf_token=userdata.get('HF_TOKEN')
child_env={n:v for n,v in os.environ.items() if all(x not in n.upper() for x in ('TOKEN','KEY','SECRET'))}; child_env['CEG_WM_ROOT_KEY']=root_key; child_env['HF_TOKEN']=hf_token
try:
 p=subprocess.run([sys.executable,'-m','experiments.run_geometry_v1_qk_operational_preflight','--repo-root',str(repo),'--expected-exact',EXECUTION_EXACT,*paths],cwd=repo,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL,check=False,timeout=1800)
 lines=p.stdout[:4096].decode('utf-8','strict').splitlines()
 if len(lines)!=1 or len(p.stdout)>4096: raise RuntimeError('bounded receipt')
 payload=json.loads(lines[0].split(' ',1)[1]); work=pathlib.Path('/content/geometry-receipt'); work.mkdir(exist_ok=False)
 terminal='success.json' if p.returncode==0 else 'failure.json'
 for name,value in [('receipt.json',payload),(terminal,{'status':payload.get('status')} )]:
  with (work/name).open('x',encoding='utf-8') as h: json.dump(value,h,sort_keys=True)
 manifest={'execution_exact':EXECUTION_EXACT,'run_id':RUN_ID,'files':['receipt.json',terminal,'manifest.json','SHA256SUMS']}
 with (work/'manifest.json').open('x',encoding='utf-8') as h: json.dump(manifest,h,sort_keys=True)
 sums=''.join(hashlib.sha256((work/name).read_bytes()).hexdigest()+'  '+name+'\n' for name in ['receipt.json',terminal,'manifest.json'])
 with (work/'SHA256SUMS').open('x',encoding='ascii') as h: h.write(sums)
 archive=pathlib.Path('/content')/(RUN_ID+'.zip')
 with zipfile.ZipFile(archive,'x') as z:
  [z.write(work/name,name) for name in ['receipt.json',terminal,'manifest.json','SHA256SUMS']]
 with archive.open('rb') as h: digest=hashlib.sha256(h.read()).hexdigest()
 sidecar=archive.with_suffix('.zip.sha256')
 with sidecar.open('xb') as h: h.write((digest+'  '+archive.name+'\n').encode())
 run_dir.mkdir(parents=True)
 for source,target in ((archive,run_dir/archive.name),(sidecar,run_dir/sidecar.name)):
  with source.open('rb') as read, target.open('xb') as write: write.write(read.read())
 print(lines[0])
finally:
 child_env.pop('CEG_WM_ROOT_KEY',None); child_env.pop('HF_TOKEN',None)
 for path in input_dir.glob('*'): path.unlink()
 input_dir.rmdir()
